# RoboCoach: train, evaluate, and export YOLO OBB

Use a prepared, reviewed YOLO OBB dataset. Full training footage and datasets are not included. Supply your own dataset or use an authorized local copy of the project data. See [the training guide](../docs/TRAINING.md) for class mappings, split limitations, and dataset layout. This notebook trains a **new** run; it does not overwrite the published checkpoints.


In [ ]:
%pip install -q ultralytics==8.4.216 pyyaml pillow


## Configure paths

In Colab, optionally mount Drive first. Set DATA_YAML to your prepared dataset configuration. Choose the matching class mode and checkpoint. The five-class checkpoint is experimental; it is not supported by the browser Toggle loader.


In [ ]:
# Optional in Colab:
# from google.colab import drive
# drive.mount("/content/drive")

from pathlib import Path

MODE = "pins"  # "toggle" or "pins"
REPO = Path.cwd()  # Set to the RoboCoach checkout directory.
DATA_YAML = Path("/content/your-reviewed-dataset/data.yaml")
CHECKPOINT = REPO / "models" / MODE / "best.pt"
PROJECT = Path("/content/drive/MyDrive/RoboCoach")
RUN_NAME = f"{MODE}-obb-new-experiment"
DEVICE = "cpu"  # Use 0 only if your runtime actually has a CUDA GPU.
EPOCHS = 50
IMGSZ = 640

assert DATA_YAML.is_file(), f"Dataset config missing: {DATA_YAML}"
assert CHECKPOINT.is_file(), f"Checkpoint missing: {CHECKPOINT}"


## Validate the class mapping

OBB label rows contain a class ID followed by four normalized corner pairs. Keep every identifiable object of the selected classes labeled in each image. A Toggle-only frame with unlabeled pins is not a complete five-class training example. Split by match/video where possible; neighboring frames are correlated.


In [ ]:
import yaml

config = yaml.safe_load(DATA_YAML.read_text())
names = config["names"]
names = list(names.values()) if isinstance(names, dict) else names
expected = ["toggle"] if MODE == "toggle" else ["toggle", "red_pin", "blue_pin", "yellow_pin", "cup"]
assert [n.lower() for n in names] == expected, f"Class mapping mismatch: {names}"
assert "train" in config and "val" in config, "Separate train and val splits are required."
print("Classes:", names)


In [ ]:
from ultralytics import YOLO

model = YOLO(str(CHECKPOINT))
results = model.train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, device=DEVICE,
    project=str(PROJECT), name=RUN_NAME, exist_ok=False, seed=0,
)
run_dir = Path(model.trainer.save_dir)
best = run_dir / "weights/best.pt"
print("Run:", run_dir)
print("Best checkpoint:", best)


## Evaluate the selected checkpoint

results.csv records epoch-by-epoch metrics. Evaluate best.pt explicitly for best-checkpoint metrics; those need not equal the final CSV row. Precision/recall and mAP measure detection, not match-scoring accuracy.


In [ ]:
trained = YOLO(str(best))
metrics = trained.val(data=str(DATA_YAML), imgsz=IMGSZ, device=DEVICE)
print(metrics.results_dict)


In [ ]:
from IPython.display import Image, display

for name in ["results.png", "val_batch0_labels.jpg", "val_batch0_pred.jpg"]:
    image = run_dir / name
    if image.exists():
        print(name)
        display(Image(filename=str(image)))


## Export a browser-compatible Toggle model

Only the single-class Toggle export is currently accepted by RoboCoach. Do not load the five-class pins/cups model into its single-class loader.


In [ ]:
if MODE == "toggle":
    onnx_path = trained.export(
        format="onnx", imgsz=640, opset=17, dynamic=False,
        simplify=True, nms=False, device="cpu",
    )
    print("ONNX:", onnx_path)
else:
    print("Five-class browser integration is planned; keep best.pt for evaluation and future training.")
